# Phase 2: GCN and GAT Baselines
Trains GCN and GAT on Twitter15 and Twitter16 with 5 random seeds on a fixed 60/20/20 stratified split.
Reports mean ± std macro-F1 and accuracy on the test set.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import torch

from data.dataset import CascadeDataset
from models.gcn import GCNClassifier
from models.gat import GATClassifier
from models.bigcn import BiGCNClassifier
from training.trainer import run_experiment

DATA_ROOT = "../Twitter15_Twitter16"

# MPS has high kernel-launch overhead for many small graphs; CPU or CUDA preferred
if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(f"Device: {DEVICE}")

In [11]:
ds15 = CascadeDataset(root=DATA_ROOT, name="twitter15")
ds16 = CascadeDataset(root=DATA_ROOT, name="twitter16")
IN_CHANNELS = ds15[0].x.shape[1]  # 30
print(f"Twitter15: {len(ds15)} graphs   Twitter16: {len(ds16)} graphs   in_channels: {IN_CHANNELS}")

Twitter15: 1490 graphs   Twitter16: 818 graphs   in_channels: 30


In [ ]:
SEEDS = [0, 1, 2, 3, 4]
EPOCHS = 200

GCN_KWARGS   = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, dropout=0.1)
GAT_KWARGS   = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, heads=4, dropout=0.1)
BIGCN_KWARGS = dict(in_channels=IN_CHANNELS, hidden_channels=128, num_layers=3, dropout=0.1)

In [ ]:
print("=" * 50)
print("GCN — Twitter15")
print("=" * 50)
gcn15 = run_experiment(
    GCNClassifier, GCN_KWARGS, ds15, SEEDS,
    epochs=EPOCHS, warmup_ratio=0.1, device=DEVICE,
    results_dir="../results", model_name="gcn", dataset_name="twitter15",
)

import json, os
os.makedirs("../results", exist_ok=True)
with open("../results/gcn_twitter15.json", "w") as f:
    json.dump(gcn15, f, indent=2)

In [ ]:
print("=" * 50)
print("GAT — Twitter15")
print("=" * 50)
gat15 = run_experiment(
    GATClassifier, GAT_KWARGS, ds15, SEEDS,
    epochs=EPOCHS, warmup_ratio=0.1, device=DEVICE,
    results_dir="../results", model_name="gat", dataset_name="twitter15",
)

with open("../results/gat_twitter15.json", "w") as f:
    json.dump(gat15, f, indent=2)

In [ ]:
print("=" * 50)
print("GCN — Twitter16")
print("=" * 50)
gcn16 = run_experiment(
    GCNClassifier, GCN_KWARGS, ds16, SEEDS,
    epochs=EPOCHS, warmup_ratio=0.1, device=DEVICE,
    results_dir="../results", model_name="gcn", dataset_name="twitter16",
)

with open("../results/gcn_twitter16.json", "w") as f:
    json.dump(gcn16, f, indent=2)

In [ ]:
print("=" * 50)
print("GAT — Twitter16")
print("=" * 50)
gat16 = run_experiment(
    GATClassifier, GAT_KWARGS, ds16, SEEDS,
    epochs=EPOCHS, warmup_ratio=0.1, device=DEVICE,
    results_dir="../results", model_name="gat", dataset_name="twitter16",
)

with open("../results/gat_twitter16.json", "w") as f:
    json.dump(gat16, f, indent=2)

In [ ]:
print("=" * 50)
print("BiGCN — Twitter15")
print("=" * 50)
bigcn15 = run_experiment(
    BiGCNClassifier, BIGCN_KWARGS, ds15, SEEDS,
    epochs=EPOCHS, warmup_ratio=0.1, device=DEVICE,
    results_dir="../results", model_name="bigcn", dataset_name="twitter15",
)

with open("../results/bigcn_twitter15.json", "w") as f:
    json.dump(bigcn15, f, indent=2)

In [ ]:
print("=" * 50)
print("BiGCN — Twitter16")
print("=" * 50)
bigcn16 = run_experiment(
    BiGCNClassifier, BIGCN_KWARGS, ds16, SEEDS,
    epochs=EPOCHS, warmup_ratio=0.1, device=DEVICE,
    results_dir="../results", model_name="bigcn", dataset_name="twitter16",
)

with open("../results/bigcn_twitter16.json", "w") as f:
    json.dump(bigcn16, f, indent=2)

In [ ]:
import json, os

def load_result(path):
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

_gcn15   = load_result("../results/gcn_twitter15.json")   or gcn15
_gat15   = load_result("../results/gat_twitter15.json")   or gat15
_bigcn15 = load_result("../results/bigcn_twitter15.json") or bigcn15
_gcn16   = load_result("../results/gcn_twitter16.json")   or gcn16
_gat16   = load_result("../results/gat_twitter16.json")   or gat16
_bigcn16 = load_result("../results/bigcn_twitter16.json") or bigcn16

print("\n" + "=" * 62)
print(f"{'Model':<10} {'Dataset':<12} {'Acc mean±std':>16} {'Macro-F1 mean±std':>20}")
print("-" * 62)

rows = [
    ("GCN",   "Twitter15", _gcn15),
    ("GAT",   "Twitter15", _gat15),
    ("BiGCN", "Twitter15", _bigcn15),
    ("GCN",   "Twitter16", _gcn16),
    ("GAT",   "Twitter16", _gat16),
    ("BiGCN", "Twitter16", _bigcn16),
]
for model, dataset, res in rows:
    if res is None:
        print(f"{model:<10} {dataset:<12}  (no results yet)")
        continue
    acc = f"{res['test_acc_mean']:.3f} ± {res['test_acc_std']:.3f}"
    f1  = f"{res['test_f1_mean']:.3f} ± {res['test_f1_std']:.3f}"
    print(f"{model:<10} {dataset:<12} {acc:>16} {f1:>20}")

print("=" * 62)
print("(5 seeds, 60/20/20 stratified split, best val-F1 checkpoint)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = [f"{m}\n{d.replace('Twitter','TW')}" for m, d, _ in rows if _ is not None]
means  = [r["test_f1_mean"] for _, _, r in rows if r is not None]
stds   = [r["test_f1_std"]  for _, _, r in rows if r is not None]
colors = ["#7f8c8d", "#3498db", "#e67e22", "#7f8c8d", "#3498db", "#e67e22"][:len(labels)]

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(labels))
bars = ax.bar(x, means, yerr=stds, capsize=5, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(0.25, color="#e74c3c", linestyle="--", linewidth=1, label="random chance")
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Macro-F1")
ax.set_title("Baseline results (mean ± std, 5 seeds)")
ax.set_ylim(0, 1.0)
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, f"{m:.3f}",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
os.makedirs("../results", exist_ok=True)
plt.savefig("../results/baseline_results.png", dpi=120, bbox_inches="tight")
plt.show()